# Build LLM from Scratch

In [2]:
from jedi.third_party.typeshed.stubs.docutils.docutils.utils.math.math2html import Position
from torch.utils.data import DataLoader

from src.GPTDatasetV1 import GPTDatasetV1

# Download the text data. Comment out the following cell if you already have the file saved locally.
file_path = "data/the-verdict.txt"
'''
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
urllib.request.urlretrieve(url, file_path)
'''

'\nimport urllib.request\nurl = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")\nurllib.request.urlretrieve(url, file_path)\n'

In [4]:
# Read the text file
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters: ", len(raw_text))
print(raw_text[:100])

Total number of characters:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [5]:
# Example tokenization
import re
text = "Hello, world! This is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world!', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [6]:
# Segments punctuation example
result = re.split(r'([,.!?]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '!', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [14]:
# remove empty strings / whitespace.
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '.']


Note: You may not want to remove whitespaces if formatting is important. For example, if you want to preserve line breaks or if you are processing python code which uses whitespace for indentation.

Additionally, we leave capitalization intact to ensure proper nouns are identified. With enough data and training, the probabilities will ensure the correct capitalization is used.

In [15]:
# Let's modify to handle a more complex example
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [24]:
from src.DataProcessing import tokenizer_v1
preprocessed_tokens = tokenizer_v1(raw_text)
print(f"Token count: {len(preprocessed_tokens)}")
print(preprocessed_tokens[:30])

Token count: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [25]:
all_words = sorted(set(preprocessed_tokens))
vocab_size: int = len(all_words)
print(f"Vocab size: {vocab_size}")

Vocab size: 1130


In [26]:
# create a dictionary mapping words to integers
vocab = {token: integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(f"{i}: {item}")
    if i >= 30:
        break

0: ('!', 0)
1: ('"', 1)
2: ("'", 2)
3: ('(', 3)
4: (')', 4)
5: (',', 5)
6: ('--', 6)
7: ('.', 7)
8: (':', 8)
9: (';', 9)
10: ('?', 10)
11: ('A', 11)
12: ('Ah', 12)
13: ('Among', 13)
14: ('And', 14)
15: ('Are', 15)
16: ('Arrt', 16)
17: ('As', 17)
18: ('At', 18)
19: ('Be', 19)
20: ('Begin', 20)
21: ('Burlington', 21)
22: ('But', 22)
23: ('By', 23)
24: ('Carlo', 24)
25: ('Chicago', 25)
26: ('Claude', 26)
27: ('Come', 27)
28: ('Croft', 28)
29: ('Destroyed', 29)
30: ('Devonshire', 30)


When we want to turn token ID back into text, we need an inverse version of the vocabulary for efficient lookups in the other direction.

To ensure reproducibility, we will create a class for the tokenizer.

In [17]:
from src.Tokenizers.SimpleTokenizerV1 import SimpleTokenizerV1

tokenizer = SimpleTokenizerV1(vocab)
text = """"
It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride.
"""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [18]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [19]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))

KeyError: 'Hello'

Notice: that we get a key error if there is a word found not in the vocabulary. There are three common ways to deal with this: (1) Add special context tokens such as <|unk|> or (2) Add a mask token to ignore words that are not in the vocabulary, or (3) use a better tokenizer that can break down / still represent unseen words (byte pair encoding).

### Enable Tokenizer to handle unknown tokens

In [30]:
from src.Tokenizers.SimpleTokenizerV2 import SimpleTokenizerV2
all_tokens = sorted(set(preprocessed_tokens))
all_tokens.extend(["<|endoftext|>", "<|unk|>"]) # Add special tokens
vocab = {token: integer for integer, token in enumerate(all_tokens)}
print(f"Token count: {len(vocab)}")

Token count: 1132


Notice there are two additional token in our vocab from earlier.

In [41]:
# Now let's try out our new tokenizer.
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
tokenizer = SimpleTokenizerV2(vocab)
print(text)
print(tokenizer.encode(text))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


Notice that the code no longer errors out when unknown tokens are seen.

In [42]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


Some researchers also consider additional special tokens such as:
[BOS] (beginning of sequence)
[EOS] (end of sequence)
[PAD] (padding)

### Byte Pair Encoding (Alternative Tokenization Methodology)

In [44]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 5372, 13]


In [45]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace.


Note: Rather than relying on the unknown token. Byte pair encoding breaks down words that are not in it's vocabulary.

Data sampling with sliding window.
Aka codification of the self-supervised learning data set generation process.

In [47]:
enc_text = tokenizer.encode(raw_text)
print(f"Token length: {len(enc_text)}")

Token length: 5145


In [50]:
enc_sample = enc_text[50:]
contex_size = 4
x = enc_sample[:contex_size]
y = enc_sample[1:contex_size+1]
print(f"Context size: {contex_size}")
print(f"Input: {x}")
print(f"Output:     {y}")

Context size: 4
Input: [290, 4920, 2241, 287]
Output:     [4920, 2241, 287, 257]


In [51]:
for i in range(1, contex_size+1):
    contex = enc_sample[:i]
    desired = enc_sample[i]
    print(contex, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


Everything to the left of the arrow represents the input to the LLM, and the token Id on the right side represents the token the LLM is supposed to predict.

In [53]:
for i in range(1, contex_size+1):
    contex = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(contex), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


For efficient data loading, we use PyTorch's built-in Dataset and Data Loader classes.

In [58]:
from GPTDatasetV1 import GPTDatasetV1
from torch.utils.data import DataLoader

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True,
                         drop_last=True, number_of_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=number_of_workers
    )
    return dataloader

In [59]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


The first tensor stores the input token Ids, and the second tensor stores the target token Ids. Since max length is set to 4, each two tensors contains four token Ids. Note: an input of 4 is quite small. It is common to train LLMs with input sizes of at least 256.

In [60]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


Looking at the second tensor above we start to see the pattern and implications of stride length. The stride dictates the number of positions the inputs shift across batches, emulating a sliding window.

Note: Small batch sizes require less memory during training but lead to more noisy model updates. Batch size is a tradeoff and hyperparameter to experiment with when training LLMs.

In [61]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs )
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


Increasing the stride to 4 to utilize the data set fully. We don't skip a single word. This avoids any overlap between the batches since more overlap could lead to increased overfitting.

### Creating token embeddings

In [65]:
import torch
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


The weight matrix of the embedding layer contains small, random values. These values are optimized during LLM training as part of the LLM optimization itself. Note: there are 6 rows. One for each of the six possible tokens in the vocabulary. There are three columns one for each of the embedding dimensions.

In [66]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


Notice, at index 3, the fourth row is returned. For those who are familiar with one-hot encoding, the embedding approach is just an efficient implementation of one-hot-encoding followed by matrix multiplication in a fully connected layer. Since it is a layer in a neural network layer, it's values can be optimized via backpropagation. We can get embedding values via a simple lookup operation as demonstrated directly above.

### Encoding word position

The shortcoming of the self-attention mechanism is that it doesn't have a notion of position or order for the tokens within a sequence.

Note: Modern LLMs use RoPE (Rotary Position Embedding) to address this issue. Rather than the simple method described below. The original transformer paper used sinusoidal positional encodings (rotations), but this turned out to be less than optimal since different frequencies had different implications. Think of it as many clocks running at different speeds. RoPE was proposed so that rotations contain 100% of the positional information and the attention between two tokens naturally reflects their distance from each other. So RoPE alters the relationship between tokens based on their positions without directly modifying the content being passed through the value vectors.

The single most important intuition is: RoPE turns position into an angle.

Note on similiary: the model does not require rotated Q/K vectors to preserve pure semantic similarity. Their job is not to be a semantic embedding space. Their job is to produce useful attention scores.

That is probably the central conceptual distinction:

Embedding similarity != Attention similarity

An embedding space tries to represent meaning.

An attention space tries to represent relevance between tokens in context.

RoPE intentionally injects position into that relevance calculation.

Why have separate Q and K matrices?
Separate Q and K turn attention from similarity into compatibility.
That is the conceptual reason.
And mathematically, the distinction is important because with separate matrices,
the model can learn an asymmetric, directional relationship between two tokens instead of only asking whether they resemble each other.

There are two broad categories of position-aware embeddings: relative position embedding and absolute position embedding. Relative position embedding is generally between since the model will learn to generalize better to sequences of varying length, even if it hasn't seen lengths during training. However, OpenAI GPT models have used absolute positional embedding that are optimized during the training process rather than being fixed or predefined. (By this, I think the book is referring to RoPE. RoPE uses the absolute position to determine how much to rotate the vectors, but the relative positions can be computed from the attention relationship. So absolute position goes in, but the model handles the relative calculations. There are not fixed or predefined as they were in the original transformer paper with absolute positional embeddings.)

Now let's create a more realistic and useful embedding size and encode the input tokens into a 256 dimension vector representation. (In GPT-3, the embedding size is 12,288 dimensions), but still reasonable for experienetation. Also byte pair encoding had a vocabulary size of 50,257:

In [67]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [68]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs )
print("\nInput shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Input shape:
 torch.Size([8, 4])


The data batch consists of eight text samples with four tokens each. Now let's use the embedding layer to embed these token IDs into 256-dimensional vectors:

In [69]:
token_embeddings = token_embedding_layer(inputs)
print("\nToken embeddings shape:\n", token_embeddings.shape)


Token embeddings shape:
 torch.Size([8, 4, 256])


Now to implement the GPT model's original absolute embedding approach, we just need to create another embedding layer than has the same dimensions but instead holds the positional embedding transformations.

In [71]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("Positional Embedding shape:",pos_embeddings.shape)

Positional Embedding shape: torch.Size([4, 256])


We now see that we have positional embeddings consisting of four 256-dimensional vectors. This gives use a 256-dimensional position embedding vector we can use to transform the token embedding at each of the 4 possible positions.

In [72]:
input_embeddings = token_embeddings + pos_embeddings
print("Shape of position transformed token embeddings:", input_embeddings.shape)

Shape of position transformed token embeddings: torch.Size([8, 4, 256])


#### In Summary:
- LLMs require textual data to be converted into numerical vectors known as embeddings since they can't process raw text. Embeddings transform discrete data (like text or images) into continuous vector spaces, making them compatible with neural network operations.
- Text is broken into tokens, which can be words or characters. Then the tokens are converted into integer representations, termed token Ids.
- We use sliding window approach on tokenized data to generate input-target pairs for LLM training.
- Embedding layers in PyTorch function as lookup operations corresponding to token IDs. The resulting embedding vectors provide continuous representations of tokens, which is crucial for training deep learning models.
- While token embeddings provide consistent vector representations for each token, they lack a sense of the token's position in a sentence. To rectify this, two main types of positional embeddings exist: absolute and relative.